In [ ]:
#=================================================================
# Celda 1 — Setup + cargar parquets raw
#=================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

RAW_DIR    = Path("output/economico/01-raw")
SILVER_DIR = Path("output/economico/02-silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

parquets = sorted(RAW_DIR.glob("*.parquet"))
assert len(parquets) > 0, f"❌ No hay parquets en {RAW_DIR}"

dfs_raw = {}
for p in parquets:
    dfs_raw[p.stem] = pd.read_parquet(p)
    print(f"✅ {p.stem}: {dfs_raw[p.stem].shape}")

In [ ]:
#=================================================================
# Celda 2 — Tasa de paro por CCAA × año
# raw_tasa_paro_ccaa: dim_0=tipo_dato, dim_1=sexo, dim_2=ccaa, dim_3=edad
# año extraído de 'fecha' (no existe col 'anyo' en este parquet)
#=================================================================
df_paro = dfs_raw["raw_tasa_paro_ccaa"].copy()

df_paro_clean = (
    df_paro
    .rename(columns={"dim_0": "tipo_dato", "dim_1": "sexo",
                     "dim_2": "CCAA", "dim_3": "grupo_edad"})
    .assign(año=lambda x: pd.to_numeric(x["fecha"].str[:4], errors="coerce").astype("Int64"))
    .query(
        "tipo_dato.str.contains('paro', case=False, na=False) and "
        "sexo == 'Ambos sexos' and "
        "grupo_edad == 'Total' and "
        "CCAA != 'Total Nacional' and "
        "CCAA == CCAA and "
        "secreto == False"
    )
    .dropna(subset=["valor"])
    .groupby(["CCAA", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "tasa_paro"})
    .dropna(subset=["CCAA", "año"])
)

print(f"✅ tasa_paro: {df_paro_clean.shape}")
print(f"   Años: {sorted(df_paro_clean['año'].dropna().unique())}")
print(f"   CCAA: {df_paro_clean['CCAA'].nunique()}")
print(df_paro_clean.head())

In [ ]:
#=================================================================
# Celda 3 — IPV precio vivienda anual (nacional × año)
# raw_ipv_precio_vivienda_anual: col 'anyo' ya existe como int
# Filtro: índices_y_tasas=='Media anual', general=='General'
#=================================================================
df_ipv = dfs_raw["raw_ipv_precio_vivienda_anual"].copy()

df_ipv_clean = (
    df_ipv
    .rename(columns={
        "índices_y_tasas": "indicador",
        "general,_vivienda_nueva_y_de_segunda_mano": "tipo_vivienda"
    })
    .query(
        "indicador == 'Media anual' and "
        "tipo_vivienda == 'General' and "
        "secreto == False"
    )
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .dropna(subset=["valor"])
    .groupby("año", as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "ipv_vivienda"})
)

print(f"✅ ipv_vivienda: {df_ipv_clean.shape}")
print(f"   Años: {sorted(df_ipv_clean['año'].dropna().unique())}")
print(df_ipv_clean.head())

In [ ]:
#=================================================================
# Celda 4 — Compraventas de vivienda por CCAA × año
# raw_compraventas_vivienda: col 'comunidades_y_ciudades_autónomas', 'anyo'
# Filtro: estado_de_la_vivienda=='General', tipo_de_dato=='Número'
#=================================================================
df_cv = dfs_raw["raw_compraventas_vivienda"].copy()

df_cv_clean = (
    df_cv
    .rename(columns={"comunidades_y_ciudades_autónomas": "CCAA"})
    .query(
        "estado_de_la_vivienda == 'General' and "
        "tipo_de_dato == 'Número' and "
        "CCAA != 'Total Nacional' and "
        "CCAA == CCAA and "
        "secreto == False"
    )
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .dropna(subset=["valor"])
    .groupby(["CCAA", "año"], as_index=False)["valor"]
    .sum()
    .rename(columns={"valor": "compraventas"})
    .dropna(subset=["CCAA", "año"])
)

print(f"✅ compraventas: {df_cv_clean.shape}")
print(f"   Años: {sorted(df_cv_clean['año'].dropna().unique())}")
print(f"   CCAA: {df_cv_clean['CCAA'].nunique()}")
print(df_cv_clean.head())

In [ ]:
#=================================================================
# Celda 5 — Merge final CCAA × año
#=================================================================
df_eco = pd.merge(df_paro_clean, df_cv_clean, on=["CCAA", "año"], how="outer")
df_eco = pd.merge(df_eco, df_ipv_clean, on="año", how="left")

df_eco["CCAA"] = df_eco["CCAA"].str.strip()
df_eco["año"]  = df_eco["año"].astype("Int64")
df_eco = df_eco.sort_values(["CCAA", "año"]).reset_index(drop=True)

print(f"📊 Panel económico final: {df_eco.shape}")
print(f"   Años: {sorted(df_eco['año'].dropna().unique())}")
print(f"   CCAA: {df_eco['CCAA'].nunique()}")
print(f"\n   Nulos por columna:\n{df_eco.isnull().sum()}")
print(f"\n{df_eco.head(10).to_string()}")

In [ ]:
#=================================================================
# Celda 6 — Export
#=================================================================
ruta_parquet = SILVER_DIR / "economico_ccaa_año.parquet"
ruta_csv     = SILVER_DIR / "economico_ccaa_año.csv"

df_eco.to_parquet(ruta_parquet, index=False)
df_eco.to_csv(ruta_csv, index=False)

print(f"✅ Guardado en: {SILVER_DIR}")
print(f"   Shape final: {df_eco.shape}")
print(f"   Columnas: {list(df_eco.columns)}")